In [1]:
import pandas as pd
import os
from tqdm import tqdm
import yfinance as yf

In [2]:
def get_stock_splits(ticker_symbol):
    # Create the Ticker object
    stock = yf.Ticker(ticker_symbol)
    
    # Retrieve the splits (Returns a Pandas Series)
    splits = stock.splits
    
    # Check if there are any splits
    if splits.empty:
        print(f"No split data found for {ticker_symbol}")
        return None
        
    return splits

# Example: Get splits for Apple (AAPL) or Nvidia (NVDA)
ticker = "BIIB"
split_data = get_stock_splits(ticker)

if split_data is not None:
    print(f"--- Split History for {ticker} ---")
    # Print the last 5 splits
    print(split_data.tail(5))
    
    # Example: Accessing a specific split date programmatically
    # Convert index to datetime to ensure easy filtering
    last_split_date = split_data.index[-1]
    last_split_ratio = split_data.iloc[-1]
    
    print(f"\nMost recent split was on {last_split_date.date()} with a ratio of {last_split_ratio}:1")

--- Split History for BIIB ---
Date
1999-12-21 00:00:00-05:00    2.0
2001-01-18 00:00:00-05:00    3.0
Name: Stock Splits, dtype: float64

Most recent split was on 2001-01-18 with a ratio of 3.0:1


In [3]:
# initialization
from tickers import NAS_100_TICKER

tickers = NAS_100_TICKER
time_interval = "1_min" # "1_hour", "15_mins", "1_min"
parent_dir = "/Volumes/Elements/stock_market_data/nasdaq_100_2025_constituents/"
processed_dir = f"/Volumes/Elements/stock_market_data/nasdaq_100_2025_constituents_processed/{time_interval}/"

for ticker in tickers:
    print(f"\n---------- Processing ticker: {ticker} ----------")

    # load data
    data_dir = os.path.join(parent_dir, time_interval)
    for file_name in os.listdir(data_dir):
        if file_name.startswith(ticker) and file_name.endswith(".csv"):
            file_path = os.path.join(data_dir, file_name)
            break

    df = pd.read_csv(file_path, index_col=0, header=[0,1])
    df.index = pd.to_datetime(df.index)
    # convert index to US Eastern Time
    df.index = df.index.tz_localize('UTC').tz_convert('America/New_York')


    # ---------- Filter based on market opening hours ----------
    # The US market opens at 14:30 UTC and closes at 21:00 UTC
    # The US market opens at 9:30 and closes at 16:00 Eastern Time
    filter_start_end_time = {"1_hour": ['9:00', '15:00'],
                            "15_mins": ['9:30', '15:45'],
                            "1_min": ['9:30', '15:59']}
    # include_start=True and include_end=True are default
    filtered_df = df.between_time(filter_start_end_time[time_interval][0], filter_start_end_time[time_interval][1])


    # ---------- Detect inconsistent timestamps ----------
    # loop through the index and check the time difference between consecutive timestamps
    time_diffs = filtered_df.index.to_series().diff().dropna()
    inconsistent_timestamps = time_diffs[time_diffs > pd.Timedelta(days=5)]
    if not inconsistent_timestamps.empty:
        print(f"Inconsistent timestamps found for {ticker} in {time_interval} data:")
        print(inconsistent_timestamps)
    else:
        print(f"No inconsistent timestamps found for {ticker} in {time_interval} data.\n")

    # split the data into chunks based on inconsistent timestamps
    data_chunks = []
    if not inconsistent_timestamps.empty:
        split_indices = inconsistent_timestamps.index
        prev_index = filtered_df.index[0]
        for split_index in split_indices:
            chunk = filtered_df.loc[prev_index:split_index - pd.Timedelta(minutes=1)]
            data_chunks.append(chunk)
            prev_index = split_index
        # add the last chunk
        last_chunk = filtered_df.loc[prev_index:]
        data_chunks.append(last_chunk)
    else:
        data_chunks = [filtered_df]


    # ---------- Detect unadjusted price splits for each chunk ----------
    for i, chunk in enumerate(data_chunks):
        chunk_start_date = chunk.index.min().strftime('%Y-%m-%d')
        chunk_end_date = chunk.index.max().strftime('%Y-%m-%d')
        print(f"\nAnalyzing chunk {i+1} from {chunk_start_date} to {chunk_end_date}...")
        
        # Check for unadjusted splits in 'Close' prices
        close_prices = chunk[('Close', ticker)]
        price_diffs = close_prices.pct_change().dropna()
        
        # Define a threshold for detecting splits (e.g., a change greater than 50%)
        split_threshold = 0.5
        potential_splits = price_diffs[abs(price_diffs) > split_threshold]

    if not potential_splits.empty:
        print(f"Potential unadjusted splits detected in chunk {i+1}:")
        for date, change in potential_splits.items():
            print(f"  Date: {date.date()}, Price Change: {change:.2%}")
    else:
        print(f"No unadjusted splits detected in chunk {i+1}.")


    # ---------- Save the splitted data ----------
    if len(data_chunks) > 3:
        print(f"WARNING: {ticker} has {len(data_chunks)} data chunks due to inconsistent timestamps. Skipping...")
    elif not potential_splits.empty:
        print(f"WARNING: {ticker} has potential unadjusted price splits detected. Skipping...")
    else:
        for i, chunk in enumerate(data_chunks):
            chunk_start_date = chunk.index.min().strftime('%Y-%m-%d')
            chunk_end_date = chunk.index.max().strftime('%Y-%m-%d')
            chunk_file_name = f"{ticker}_{time_interval}_{chunk_start_date}_to_{chunk_end_date}.csv"
            chunk_file_path = os.path.join(processed_dir, chunk_file_name)
            chunk.to_csv(chunk_file_path)
            print(f"Saved chunk to {chunk_file_path}")


---------- Processing ticker: AAPL ----------
No inconsistent timestamps found for AAPL in 1_min data.


Analyzing chunk 1 from 2015-12-07 to 2025-12-01...
No unadjusted splits detected in chunk 1.
Saved chunk to /Volumes/Elements/stock_market_data/nasdaq_100_2025_constituents_processed/1_min/AAPL_1_min_2015-12-07_to_2025-12-01.csv

---------- Processing ticker: ABNB ----------
No inconsistent timestamps found for ABNB in 1_min data.


Analyzing chunk 1 from 2020-12-10 to 2025-12-01...
No unadjusted splits detected in chunk 1.
Saved chunk to /Volumes/Elements/stock_market_data/nasdaq_100_2025_constituents_processed/1_min/ABNB_1_min_2020-12-10_to_2025-12-01.csv

---------- Processing ticker: ADBE ----------
No inconsistent timestamps found for ADBE in 1_min data.


Analyzing chunk 1 from 2015-12-07 to 2025-12-01...
No unadjusted splits detected in chunk 1.
Saved chunk to /Volumes/Elements/stock_market_data/nasdaq_100_2025_constituents_processed/1_min/ADBE_1_min_2015-12-07_to_2025-12-01